2.基于代码一生成的向量库实现RAG
以下代码完成了RAG流程：初始化系统 → 用户提问 → 向量检索 → 构建提示词 → LLM生成答案。替换其中的API_key，运行代码，输出最终问题的答案。

API_key获取：进入 https://bailian.console.aliyun.com/?spm=5176.29597918.nav-v2-dropdown-menu-0.d_main_2_0_1.52007b08KzGnZU&tab=model&scm=20140722.M_10904465._.V_1#/model-market ，完成登入后，点击左下角“密钥管理”，点击“创建API KEY”，获得密钥。

In [ ]:
import torch
import numpy as np
import pickle
from transformers import BertTokenizer, BertModel
from openai import OpenAI
from typing import List, Dict
import time


def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """计算余弦相似度"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2) + 1e-8)


class ChunkBasedRAGSystem:
    def __init__(self, vector_db_path: str, model_path: str = "bert-base-chinese-local"):
        """初始化RAG系统（基于分块的向量库）"""
        print("=" * 60)
        print("初始化分块RAG问答系统")
        print("=" * 60)

        # 加载向量库
        print("1. 加载向量库...")
        start_time = time.time()
        
        # 直接加载原始向量库
        with open(vector_db_path, 'rb') as f:
            self.vector_db = pickle.load(f)
        
        print(f"   加载完成，共 {len(self.vector_db)} 个文档，耗时: {time.time() - start_time:.2f}秒")

        # 加载BERT模型用于查询向量化
        print("2. 加载BERT模型...")
        start_time = time.time()
        self.tokenizer = BertTokenizer.from_pretrained(model_path)
        self.model = BertModel.from_pretrained(model_path)
        self.model.eval()
        print(f"   模型加载完成，耗时: {time.time() - start_time:.2f}秒")

        # 初始化OpenAI客户端
        print("3. 初始化API客户端...")
        self.client = OpenAI(
            api_key="sk-07ba5bd13252402d94aad91eb52473cb",#替换为你的API_key,
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
        )

        # 准备向量数据
        print("4. 准备向量数据...")
        start_time = time.time()
        self.chunk_embeddings = []
        self.chunks = []

        for doc_data in self.vector_db.values():
            for chunk in doc_data["chunks"]:
                self.chunk_embeddings.append(chunk["embedding"])
                self.chunks.append({
                    "doc_filename": doc_data["filename"],
                    "chunk_id": chunk["chunk_id"],
                    "chunk_text": chunk["chunk_text"],
                    "parent_doc": chunk["parent_doc"],
                    "chunk_range": chunk["chunk_range"]
                })

        self.chunk_embeddings = np.array(self.chunk_embeddings)
        total_chunks = len(self.chunks)
        print(f"   准备完成，共 {total_chunks} 个文本块，耗时: {time.time() - start_time:.2f}秒")

        print(f"系统初始化完成！\n")

    def vectorize_query(self, query: str) -> np.ndarray:
        """向量化查询文本"""
        inputs = self.tokenizer(
            query,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = self.model(**inputs)
            query_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

        return query_embedding

    def retrieve_with_embedding(self, query: str, top_k: int = 5) -> List[Dict]:
        """使用向量相似度检索最相关的文本块"""
        query_vector = self.vectorize_query(query)

        # 计算余弦相似度
        similarities = []
        for i, chunk_vector in enumerate(self.chunk_embeddings):
            sim = cosine_similarity(query_vector, chunk_vector)
            similarities.append((sim, i))

        # 排序并获取top-k
        similarities.sort(key=lambda x: x[0], reverse=True)

        relevant_chunks = []
        seen_docs = set()  # 用于去重，避免同一文档的多个块

        for sim, idx in similarities[:top_k * 2]:  # 获取更多结果以便去重
            if len(relevant_chunks) >= top_k:
                break

            chunk_info = self.chunks[idx]
            doc_name = chunk_info["doc_filename"]

            # 避免同一文档的多个块
            if doc_name in seen_docs and len(relevant_chunks) > 0:
                continue

            seen_docs.add(doc_name)

            relevant_chunks.append({
                "doc_filename": chunk_info["doc_filename"],
                "chunk_id": chunk_info["chunk_id"],
                "chunk_text": chunk_info["chunk_text"],
                "similarity": float(sim)
            })

        return relevant_chunks

    def build_prompt(self, question: str, relevant_chunks: List[Dict]) -> str:
        """构建高质量的提示词（基于文本块）"""

        # 构建上下文
        context_parts = []
        for i, chunk in enumerate(relevant_chunks, 1):
            # 使用文本块内容
            chunk_text = chunk["chunk_text"]
            context_parts.append(chunk_text)

        context = "\n\n".join(context_parts)

        # 构建提示词
        prompt = f"""你是一个专业的农业科学助手，请基于以下提供的文档内容，准确回答用户的问题。

提供的相关文档内容：
{context}

用户的问题：{question}

请根据以上文档内容，提供专业、准确、完整的回答。

现在，请开始回答："""

        return prompt

    def answer_question(self, question: str, question_num: int = None) -> str:
        """回答单个问题，返回答案"""
        if question_num:
            print(f"\n{'=' * 80}")
            print(f"问题 {question_num}: {question}")
            print('=' * 80)

        # 检索相关文本块
        relevant_chunks = self.retrieve_with_embedding(question, top_k=3)

        if not relevant_chunks:
            return "抱歉，在知识库中没有找到相关信息。"

        # 构建提示词
        prompt = self.build_prompt(question, relevant_chunks)

        # 调用LLM
        try:
            completion = self.client.chat.completions.create(
                model="deepseek-v3",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1000,
                temperature=0.1
            )
            answer = completion.choices[0].message.content
        except Exception as e:
            answer = f"调用API时出错：{str(e)}"

        return answer


def main():
    """主函数：处理10个问题"""

    # 定义10个问题
    questions = [
        "胡柚果实采后枯水发生的根本原因是什么？有哪些关键酶活性变化与其相关？",
        "套袋对提高惠民短枝红富士苹果品质有哪些具体效应？",
        "桃果实发育过程中褐变与哪些因素相关？多酚氧化酶（PPO）的热稳定性如何？",
        "柑桔体细胞杂种在抗性方面有哪些优势？请举例说明。",
        "蕉柑的起源和分类地位如何？有哪些证据支持？",
        "低温胁迫下，香蕉与大蕉在SOD活性和ABA含量上有何差异？多效唑如何处理这些差异？",
        "石榴和桃在NaCl胁迫下对Na⁺和K⁺的吸收与转运有何不同？",
        "沙田柚自交不亲和的表现机制是什么？属于哪种不亲和类型？",
        "我国果树营养研究在哪些方面取得了进展？尚存哪些问题？",
        "桃树根癌病的生物防治方法有哪些？K84菌液的防治效果如何？"
    ]

    print("开始分块RAG问答系统处理流程")
    print(f"待处理问题数: {len(questions)}")
    print()

    # 1. 初始化RAG系统
    rag_system = ChunkBasedRAGSystem("vector_db.pkl")

    # 2. 处理每个问题并打印答案
    all_answers = []
    total_start_time = time.time()

    for i, question in enumerate(questions, 1):
        # 获取答案
        answer = rag_system.answer_question(question, question_num=i)
        all_answers.append(answer)
        
        # 打印答案
        print(f"\n答案 {i}:")
        print("-" * 80)
        print(answer)
        print("-" * 80)

    total_time = time.time() - total_start_time

    # 3. 打印统计信息
    print(f"\n{'=' * 80}")
    print("所有问题处理完成！")
    print(f"总耗时: {total_time:.2f}秒")
    print(f"平均每个问题: {total_time / len(questions):.2f}秒")
    print()


if __name__ == "__main__":
    main()

C:\Users\wk_11\.conda\envs\tos\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertModel were not initialized from the model checkpoint at bert-base-chinese-local and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


开始分块RAG问答系统处理流程
待处理问题数: 10

初始化分块RAG问答系统
1. 加载向量库...
   加载完成，共 20 个文档，耗时: 0.01秒
2. 加载BERT模型...
   模型加载完成，耗时: 0.16秒
3. 初始化API客户端...
4. 准备向量数据...
   准备完成，共 328 个文本块，耗时: 0.00秒
系统初始化完成！


问题 1: 胡柚果实采后枯水发生的根本原因是什么？有哪些关键酶活性变化与其相关？

答案 1:
--------------------------------------------------------------------------------
调用API时出错：Error code: 401 - {'error': {'message': 'Incorrect API key provided. For details, see: https://help.aliyun.com/zh/model-studio/error-code#apikey-error', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}, 'request_id': 'fbdde66a-cd3d-4695-935a-7b36b0aad24d'}
--------------------------------------------------------------------------------

问题 2: 套袋对提高惠民短枝红富士苹果品质有哪些具体效应？

答案 2:
--------------------------------------------------------------------------------
调用API时出错：Error code: 401 - {'error': {'message': 'Incorrect API key provided. For details, see: https://help.aliyun.com/zh/model-studio/error-code#apikey-error', 'type': 'invalid_request_error',